# Atelier 1 — Notebook 3 : Allocation des coûts partagés (exercice)

**Objectif** : implémenter 3 stratégies de répartition des coûts non taggés et comparer leurs effets.

Pré-requis : avoir exécuté le notebook 2 (couverture de tagging). Le constat est qu'une partie du coût n'est rattachée à aucune équipe — il faut bien la répartir d'une façon ou d'une autre pour facturer en interne (showback / chargeback).

| Méthode | Principe |
|---|---|
| **Even split** | Diviser également entre toutes les équipes |
| **Proportional** | Proportionnel à la conso taggée |
| **Custom weights** | Pondération métier (ex: équipes plateforme moins ponctionnées) |

**Règle d'or** : quelle que soit la méthode, la **somme allouée doit être égale à la somme totale** (conservation des coûts).

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.float_format', '${:,.2f}'.format)

## Cellule 1 — Chargement et séparation taggé / non taggé

In [ ]:
CUR_PATH = Path('../../ressources/datasets/cur_sample.csv')
df = pd.read_csv(CUR_PATH, parse_dates=['usage_date'])
df['tag_team'] = df['tag_team'].replace('', pd.NA)

df_tagged = df[df['tag_team'].notna()].copy()
df_untagged = df[df['tag_team'].isna()].copy()

total_tagged = df_tagged['unblended_cost'].sum()
total_untagged = df_untagged['unblended_cost'].sum()
total = total_tagged + total_untagged

print(f'Coût total : ${total:,.2f}')
print(f'  - dont taggé    : ${total_tagged:,.2f} ({total_tagged / total * 100:.1f}%)')
print(f'  - dont non taggé: ${total_untagged:,.2f} ({total_untagged / total * 100:.1f}%)')
print(f"\nÉquipes identifiées : {sorted(df_tagged['tag_team'].unique())}")

## Cellule 2 — Méthode 1 : Even split (à compléter)

Le coût non taggé est divisé **également** entre toutes les équipes identifiées.

In [ ]:
def allocate_even(df_tagged: pd.DataFrame, df_untagged: pd.DataFrame) -> pd.DataFrame:
    """
    Répartit les coûts non taggés à parts égales entre toutes les équipes.
    Retourne un DataFrame [tag_team, cout_tagge, cout_alloue, cout_total].
    """
    # TODO: calculer le coût taggé par équipe (groupby + sum)
    # TODO: calculer la part non taggée par équipe = total_untagged / nb_equipes
    # TODO: assembler dans un DataFrame avec 3 colonnes de coût
    pass

result_even = allocate_even(df_tagged, df_untagged)
display(result_even)

## Cellule 3 — Méthode 2 : Proportional (à compléter)

Le coût non taggé est réparti **proportionnellement au coût taggé** de chaque équipe.

*Exemple* : si l'équipe `payments` représente 40% du coût taggé, elle reçoit 40% du coût non taggé.

In [ ]:
def allocate_proportional(df_tagged: pd.DataFrame, df_untagged: pd.DataFrame) -> pd.DataFrame:
    """
    Répartit les coûts non taggés proportionnellement aux coûts taggés par équipe.
    Retourne un DataFrame avec le coût total alloué par équipe.
    """
    # TODO: calculer le total taggé par équipe
    # TODO: calculer la part de chaque équipe (en %)
    # TODO: répartir les coûts non taggés selon ces parts
    # TODO: retourner le DataFrame consolidé
    pass

result_proportional = allocate_proportional(df_tagged, df_untagged)
display(result_proportional)

## Cellule 4 — Méthode 3 : Custom weights (à compléter)

On utilise une pondération métier. Exemple ici : les équipes `platform` ne reçoivent que 50% de leur part proportionnelle (elles fournissent un service mutualisé), le reste est repris sur les équipes produit.

In [ ]:
WEIGHTS = {
    'platform': 0.5,
    'payments': 1.0,
    'data':     1.0,
    'mobile':   1.0,
}

def allocate_weighted(df_tagged: pd.DataFrame, df_untagged: pd.DataFrame,
                     weights: dict) -> pd.DataFrame:
    """
    Allocation pondérée : chaque équipe reçoit (poids × coût taggé) / somme(poids × coût taggé)
    fois le coût non taggé. La somme totale est conservée.
    """
    # TODO: calculer le coût taggé par équipe
    # TODO: appliquer le poids (cout_pondere = cout_tagge × weight)
    # TODO: calculer la part de chaque équipe sur la base pondérée
    # TODO: répartir le coût non taggé selon ces parts
    # TODO: retourner le DataFrame consolidé
    pass

result_weighted = allocate_weighted(df_tagged, df_untagged, WEIGHTS)
display(result_weighted)

## Cellule 5 — Tests de conservation

Pour chaque méthode, la **somme allouée doit être égale au coût total** (à 1 centime près). Si un test échoue, il y a une fuite dans la répartition.

In [ ]:
def check_conservation(result: pd.DataFrame, expected_total: float, label: str) -> None:
    if result is None:
        print(f'❌ {label:<15} : fonction non implémentée')
        return
    allocated = result['cout_total'].sum()
    diff = abs(allocated - expected_total)
    status = '✅' if diff < 0.01 else '❌'
    print(f'{status} {label:<15} : alloué=${allocated:,.2f} | attendu=${expected_total:,.2f} | écart=${diff:,.4f}')

check_conservation(result_even,         total, 'Even split')
check_conservation(result_proportional, total, 'Proportional')
check_conservation(result_weighted,     total, 'Weighted')

## Cellule 6 — Comparaison des 3 méthodes

À exécuter une fois les 3 fonctions implémentées. La table montre combien chaque équipe paie selon la méthode choisie — c'est exactement ce qui se passe en réunion FinOps quand on choisit une politique de chargeback.

In [ ]:
if all(r is not None for r in [result_even, result_proportional, result_weighted]):
    compare = (
        result_even[['tag_team', 'cout_total']].rename(columns={'cout_total': 'even'})
        .merge(result_proportional[['tag_team', 'cout_total']]
               .rename(columns={'cout_total': 'proportional'}), on='tag_team')
        .merge(result_weighted[['tag_team', 'cout_total']]
               .rename(columns={'cout_total': 'weighted'}), on='tag_team')
        .sort_values('proportional', ascending=False)
    )
    display(compare)

    try:
        import plotly.express as px
        fig = px.bar(
            compare.melt(id_vars='tag_team', var_name='methode', value_name='cout'),
            x='tag_team', y='cout', color='methode', barmode='group',
            title='Coût alloué par équipe selon la méthode de répartition'
        )
        fig.show()
    except ImportError:
        pass
else:
    print('⚠️ Implémente les 3 fonctions avant de lancer la comparaison.')

## 👉 Questions de débrief

1. Quelle équipe est **avantagée** par la méthode `even` ? Pourquoi est-ce probablement injuste ?
2. Quelle équipe est **pénalisée** par la méthode `proportional` ? Est-ce légitime selon vous ?
3. Comment justifieriez-vous la pondération `platform = 0.5` auprès des autres équipes ? Quels effets pervers anticipez-vous ?
4. À votre avis, dans une vraie organisation, quel est le **vrai facteur de succès** d'une politique de chargeback : la précision du calcul, ou autre chose ?